# Centrality Toxicity Analysis in User-Reply Networks

This notebook analyzes **user-reply graphs** from multiple gaming subreddits to study how network position relates to toxic behavior.


### Overview
- **Inputs:**  
  Pre-built user-reply CSVs (`edges_<sub>.csv`, `nodes_<sub>.csv` in `../user_reply_networks/`)
- **Outputs:**  
  Centrality tables and GLM summaries in `../results/centrality/`


### Steps
1. **Build directed graphs** where each edge = a reply (`Source → Target`).
2. **Compute centrality metrics:** in/out-degree, betweenness, eigenvector.
3. **Aggregate toxicity metrics:**  
   - `toxicity_rate` (produced)  
   - `toxic_replies_rate` (received)
4. **Flag users:**
   - **Opinion leaders:** top 10% in-degree  
   - **Toxic hubs:** high betweenness + high producer toxicity  
   - **Toxic bridges:** high betweenness + high received toxicity
5. **Fit GLMs** linking toxicity proportions to centrality and activity levels.


### Notes
- Uses quantile thresholds (`Q90` betweenness, `Q75` toxicity) with activity floors (≥5 comments/replies).  
- Empty lists (e.g., no toxic hubs) are expected for small or low-toxicity subreddits.  
- GLMs are binomial (proportion + trials) for stable estimation.

### Imports and Confirguration

In [205]:
import os, json, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm

SUBS = [
    "WoW","ElderScrollsOnline","SkyChildrenoftheLight","AlbionOnline",
    "DarkSouls","LiesofP","HollowKnight","Ninesols"
]
IN_DIR   = "../user_reply_networks"
RES_DIR  = "../results/centrality"
os.makedirs(RES_DIR, exist_ok=True)

# Floors & thresholds
MIN_COMMENTS = 5           # require at least 5 authored comments to judge "producer toxicity"
MIN_REPLIES  = 5           # require at least 5 replies received to judge "bridge toxicity"
Q_BTW        = 0.90        # top-10% betweenness to be considered a hub/bridge candidate
Q_TOX_PROD   = 0.75        # top-25% among positive producers
Q_TOX_RECV   = 0.75        # top-25% among positive receivers

### Load Edges & Nodes (per subreddit)
Reads the already-exported edge and node CSVs; applies defensive casting and fills required columns.

In [207]:
def load_subreddit_tables(name):
    epath = os.path.join(IN_DIR,  f"edges_{name}.csv")
    npath = os.path.join(IN_DIR,  f"nodes_{name}.csv")
    if not (os.path.exists(epath) and os.path.exists(npath)):
        print(f"[SKIP] {name}: missing edges/nodes CSV in {IN_DIR}")
        return None, None
    edges = pd.read_csv(epath)
    nodes = pd.read_csv(npath)

    # defensive casts
    edges["Source"] = edges["Source"].astype(str)
    edges["Target"] = edges["Target"].astype(str)
    if "Weight" not in edges: edges["Weight"] = 1.0
    if "Toxic_Edge_Frac" not in edges: edges["Toxic_Edge_Frac"] = np.nan
    if "Id" not in nodes: nodes = nodes.rename(columns={"author":"Id"})
    if "comments" not in nodes: nodes["comments"] = 0
    if "toxicity_rate" not in nodes: nodes["toxicity_rate"] = np.nan

    # keep only valid users
    EXCLUDE = {"[deleted]", "AutoModerator", None, ""}
    nodes = nodes[~nodes["Id"].isin(EXCLUDE)].copy()
    edges = edges[~edges["Source"].isin(EXCLUDE) & ~edges["Target"].isin(EXCLUDE)].copy()
    return edges, nodes

### Build Directed Reply Graph
Construct a directed graph (replier → parent_author) and attach edge attributes (weight, tox_frac).

In [209]:
def build_graph(edges, nodes):
    G = nx.DiGraph()
    G.add_nodes_from(nodes["Id"].tolist())
    for r in edges.itertuples(index=False):
        G.add_edge(
            r.Source, r.Target,
            weight=float(getattr(r, "Weight", 1.0)),
            tox_frac=float(getattr(r, "Toxic_Edge_Frac", np.nan))
        )
    return G

### Centrality Helpers
Compute degree (in/out, un/weighted), betweenness, and eigenvector centralities.

In [211]:
def degree_series(G, kind="in", weighted=False):
    if weighted:
        w = "weight"
        d = dict(getattr(G, f"{kind}_degree")(weight=w)) if kind in ("in","out") else dict(G.degree(weight=w))
    else:
        d = dict(getattr(G, f"{kind}_degree")()) if kind in ("in","out") else dict(G.degree())
    return pd.Series(d, name=f"{kind}_{'w' if weighted else 'uw'}deg")

def betweenness_series(G):
    # unweighted betweenness to highlight structural brokerage
    b = nx.betweenness_centrality(G, normalized=True, weight=None)
    return pd.Series(b, name="betweenness")

def eigenvector_series(G):
    try:
        x = nx.eigenvector_centrality_numpy(G)  # for DiGraph this uses A*x=lambda*x (outgoing influence)
    except Exception:
        x = nx.eigenvector_centrality_numpy(G.to_undirected())
    return pd.Series(x, name="eigenvector")

### Toxic Replies Received (Target-side metric)
Aggregate how much toxic content each user receives from incoming edges (weighted by reply counts).

In [213]:
def toxic_replies_received(edges_df):
    # average toxicity of replies that a user RECEIVES, weighted by reply count
    inc = edges_df.dropna(subset=["Target"]).copy()
    inc["wtox"] = inc["Weight"] * inc["Toxic_Edge_Frac"].fillna(0)
    grp = inc.groupby("Target", as_index=False).agg(
        replies_received=("Weight","sum"),
        toxic_replies_received=("wtox","sum")
    )
    grp["toxic_replies_rate"] = np.where(
        grp["replies_received"]>0,
        grp["toxic_replies_received"]/grp["replies_received"], np.nan
    )
    return grp.rename(columns={"Target":"Id"})

def robust_quantile_positive(x, q, mincount=1):
    x = pd.to_numeric(x, errors="coerce")
    pos = x[x > 0]
    if len(pos) >= mincount:
        return float(pos.quantile(q))
    # fallback to overall quantile if no positives; else 1.0 so it yields empty set
    return float(x.quantile(q)) if x.notna().any() else 1.0

### GLMs: Toxicity Production & Toxic Replies Received
Run two binomial GLMs: (1) user toxic production; (2) toxic replies received, each ~ centrality + activity.

In [215]:
def run_glms(df, subname):
    """
    GLM 1: user toxic PRODUCTION (proportion) ~ centrality + activity
      - endog: p = toxic_count / comments
      - weights: comments  (trials)
    GLM 2: toxic REPLIES RECEIVED (proportion) ~ centrality + activity
      - endog: p = toxic_replies_received / replies_received
      - weights: replies_received
    """
    out_path = os.path.join(RES_DIR, f"centrality_models_{subname}.txt")
    with open(out_path, "w") as f:

        # ===== GLM 1: production
        d1 = df.copy()
        d1["comments"] = pd.to_numeric(d1["comments"], errors="coerce").fillna(0).astype(int)
        d1 = d1.loc[d1["comments"] >= MIN_COMMENTS].copy()
        d1["toxicity_rate"] = pd.to_numeric(d1["toxicity_rate"], errors="coerce")
        d1 = d1.dropna(subset=["toxicity_rate"])
        if len(d1) >= 20:
            # proportion as endog, weights = trials
            d1["p_prod"] = d1["toxicity_rate"].clip(0,1)
            f1 = "p_prod ~ in_uwdeg + out_uwdeg + betweenness + eigenvector + comments"
            try:
                m1 = sm.GLM.from_formula(
                    f1, data=d1, family=sm.families.Binomial(), freq_weights=d1["comments"]
                ).fit(maxiter=200)
                f.write("[GLM] Toxic Production (Binomial, proportion with trials)\n")
                f.write(m1.summary().as_text() + "\n\n")
            except Exception as e:
                f.write(f"[GLM] Toxic Production ERROR: {e}\n\n")
        else:
            f.write("[GLM] Toxic Production skipped: not enough rows after filters\n\n")

        # ===== GLM 2: replies received
        d2 = df.copy()
        d2["replies_received"] = pd.to_numeric(d2["replies_received"], errors="coerce").fillna(0)
        d2["toxic_replies_received"] = pd.to_numeric(d2["toxic_replies_received"], errors="coerce").fillna(0)
        d2 = d2.loc[d2["replies_received"] >= MIN_REPLIES].copy()
        if len(d2) >= 20:
            d2["p_recv"] = (d2["toxic_replies_received"]/d2["replies_received"]).clip(0,1)
            f2 = "p_recv ~ in_uwdeg + out_uwdeg + betweenness + eigenvector + comments"
            try:
                m2 = sm.GLM.from_formula(
                    f2, data=d2, family=sm.families.Binomial(), freq_weights=d2["replies_received"]
                ).fit(maxiter=200)
                f.write("[GLM] Toxic Replies Received (Binomial, proportion with trials)\n")
                f.write(m2.summary().as_text() + "\n")
            except Exception as e:
                f.write(f"[GLM] Toxic Replies ERROR: {e}\n")
        else:
            f.write("[GLM] Toxic Replies skipped: not enough rows after filters\n")

### Process One Subreddit
Build graph → compute centralities → merge with user stats → flag leaders/toxic hubs/bridges → save outputs.

In [217]:
def process_sub(name):
    edges, nodes = load_subreddit_tables(name)
    if edges is None: return

    G = build_graph(edges, nodes)

    # centralities
    in_deg_u  = degree_series(G, "in",  weighted=False)
    out_deg_u = degree_series(G, "out", weighted=False)
    in_deg_w  = degree_series(G, "in",  weighted=True)
    out_deg_w = degree_series(G, "out", weighted=True)
    btw       = betweenness_series(G)
    eig       = eigenvector_series(G)

    cent = pd.concat([in_deg_u, out_deg_u, in_deg_w, out_deg_w, btw, eig], axis=1).reset_index()
    cent = cent.rename(columns={"index":"Id"})

    # replies-received metrics (from edges)
    tr = toxic_replies_received(edges)  # Id, replies_received, toxic_replies_received, toxic_replies_rate

    # merge
    df = nodes.merge(cent, on="Id", how="left").merge(tr, on="Id", how="left")
    for c in ["in_uwdeg","out_uwdeg","betweenness","eigenvector","toxicity_rate","toxic_replies_rate","comments","replies_received"]:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")

    # Opinion leaders: top-10% in-degree (unweighted)
    thr_in = df["in_uwdeg"].quantile(0.90) if df["in_uwdeg"].notna().any() else np.inf
    df["opinion_leader"] = df["in_uwdeg"] >= thr_in

    # Robust thresholds for “toxic” labels so they don’t collapse at 0
    thr_btw  = df["betweenness"].quantile(Q_BTW) if df["betweenness"].notna().any() else np.inf
    thr_prod = robust_quantile_positive(df.loc[df["comments"] >= MIN_COMMENTS, "toxicity_rate"], Q_TOX_PROD)
    thr_recv = robust_quantile_positive(df.loc[df["replies_received"] >= MIN_REPLIES, "toxic_replies_rate"], Q_TOX_RECV)

    # Flags
    df["toxic_hub"] = (
        (df["betweenness"] >= thr_btw) &
        (df["comments"]   >= MIN_COMMENTS) &
        (df["toxicity_rate"] >= thr_prod)
    )
    df["toxic_bridge"] = (
        (df["betweenness"] >= thr_btw) &
        (df["replies_received"] >= MIN_REPLIES) &
        (df["toxic_replies_rate"] >= thr_recv)
    )

    # Save table
    out_csv = os.path.join(RES_DIR, f"centrality_{name}.csv")
    df.to_csv(out_csv, index=False)
    print(f"[OK] Saved centrality table: {out_csv}  | rows={len(df)}")

    # GLMs (no figures)
    run_glms(df, name)

    # Top lists
    def top(df_, col, k=10):
        return df_[["Id", col]].dropna().sort_values(col, ascending=False).head(k)

    print(f"\n[{name}] Top opinion leaders (in-degree ≥ 90%):")
    print(df.loc[df["opinion_leader"], ["Id","in_uwdeg","toxicity_rate","betweenness"]]
            .sort_values("in_uwdeg", ascending=False).head(10).to_string(index=False))

    print(f"\n[{name}] Top toxic hubs (btw ≥ {Q_BTW:.2f} & tox_rate ≥ Q{int(Q_TOX_PROD*100)}% among commenters≥{MIN_COMMENTS}):")
    print(top(df[df["toxic_hub"]], "betweenness").to_string(index=False))

    print(f"\n[{name}] Top toxic bridges (btw ≥ {Q_BTW:.2f} & toxic_replies_rate ≥ Q{int(Q_TOX_RECV*100)}% among receivers≥{MIN_REPLIES}):")
    print(top(df[df["toxic_bridge"]], "betweenness").to_string(index=False))

### Run All Subreddits
Loop through each subreddit and execute the pipeline. Outputs go to ../results/centrality and ../figs.

In [219]:
for s in SUBS:
    print(f"\n=== {s} ===")
    process_sub(s)

print("\n[DONE] Centrality + GLMs complete.")
print(f"Tables → {RES_DIR}")


=== WoW ===
[OK] Saved centrality table: ../results/centrality/centrality_WoW.csv  | rows=6190

[WoW] Top opinion leaders (in-degree ≥ 90%):
                  Id  in_uwdeg  toxicity_rate  betweenness
   yourresidentferal       558       0.000000     0.043393
  early_conflict_160       260       0.200000     0.000577
            turtvaiz       259       0.166667     0.035764
               ex0ll       169       0.333333     0.014375
 olvekstoneheid_2006       149       0.000000     0.047682
        philgraves75       120       0.000000     0.004720
acrobatic_airline605       100       0.000000     0.006706
              teenoc        93       0.000000     0.006927
 ugaeismyamongusname        92       0.030303     0.036460
primordial-pineapple        84       0.000000     0.008138

[WoW] Top toxic hubs (btw ≥ 0.90 & tox_rate ≥ Q75% among commenters≥5):
                  Id  betweenness
glitteringchipmunk21     0.009806
        winstonbabar     0.004362
      jaegerjaquez25     0.002988
